# 02. 흡연 여부와 건강 지표 간 상관관계 분석

> 이 노트북은 원본 통합 노트북 `소스_코드_8팀_생체_신호_기반_흡연_여부_비교_시각화_프로젝트__-_종합.ipynb` 의 관련 셀들을 주제별로 재구성한 것입니다. 코드는 원본 그대로 옮겨왔으며(실행 결과만 초기화), 새로 작성하거나 수정한 로직은 없습니다.
>
> 실행하려면 `train_dataset.csv`(Kaggle: Smoker Status Prediction Dataset)를 동일 폴더에 두어야 합니다. 데이터 파일은 저장소에 포함되어 있지 않습니다.
>
> 원본에는 서로 다른 시점에 작성된 3개의 전처리 버전이 섞여 있습니다 (자세한 내용은 `docs/03_issues_and_troubleshooting.md` 참고). 이 재구성본은 그 버전들을 **삭제하거나 하나로 합치지 않고**, 각 노트북 안에서 '버전 A/B/C'로 구분해 모두 보존했습니다.

## 버전 A — 피어슨 상관계수 + 가로 막대그래프

In [ ]:
train

In [ ]:
train.isnull().sum()

In [ ]:
train['smoking']
ts = train['smoking'].value_counts()
#흡연 비흡연 인원 수
ts.plot(kind="pie",autopct='%1.1f%%', figsize=(6,6), startangle=90,
        colors=[C_NONSMOKER, C_SMOKER])
plt.legend(['비흡연자(0)', '흡연자(1)'], loc="upper right", frameon=False)
plt.title("흡연자 비율")
plt.ylabel("")  # 불필요한 y라벨 제거
plt.show()

In [ ]:
corr = train.corr(numeric_only=True)
corr
corr = train.select_dtypes(include='number').corr()['smoking'].drop('smoking').sort_values(ascending=True)
corr
# 흡연 여부 상관계수 파악하기

In [ ]:
colors = [C_SMOKER if x >= 0 else C_NONSMOKER for x in corr.values]
corr.plot(kind="barh",color=colors, figsize=(10,6))
plt.title('흡연과 건강지표 간 상관관계', color=C_TEXT)
plt.xlabel('상관계수', color=C_TEXT)
plt.ylabel('건강지표변수', color=C_TEXT)
plt.xticks(rotation=0)
plt.grid(axis='x')
plt.show()

## 버전 B — Point-biserial 상관계수 + p-value (통계적 유의성 포함)

In [ ]:
#흡연과상관관계분석
cont_cols = [
    'age', 'height(cm)', 'weight(kg)', 'waist(cm)',
    'eyesight(left)', 'eyesight(right)',
    'systolic', 'relaxation', 'fasting blood sugar',
    'Cholesterol', 'triglyceride', 'HDL', 'LDL',
    'hemoglobin', 'serum creatinine', 'AST', 'ALT', 'Gtp', 'BMI'
]

corr_rows = []
for col in cont_cols:
    r, p = stats.pointbiserialr(train_clean['smoking'], train_clean[col])
    corr_rows.append([col, r, p])

corr_df = pd.DataFrame(corr_rows, columns=['variable', 'correlation', 'p_value'])
corr_df = corr_df.sort_values('correlation', ascending=True)

display(corr_df)

In [ ]:
#상관관계시각화
bar_colors = [COLOR_SMOKER if x > 0 else COLOR_NONSMOKER for x in corr_df['correlation']]

plt.figure(figsize=(7, 5))
plt.barh(corr_df['variable'], corr_df['correlation'], color=bar_colors)
plt.axvline(0, color=COLOR_TEXT, linewidth=1)

plt.title('흡연과 건강지표 간 상관관계', color=COLOR_TEXT)
plt.xlabel('상관계수', color=COLOR_TEXT)
plt.ylabel('변수', color=COLOR_TEXT)
plt.grid(axis='x', linestyle='--', color=COLOR_GRID, alpha=0.6)
plt.tight_layout()
plt.show()

## 버전 C — 단순 상관계수 산출

In [ ]:
# 흡연 여부와 다른 항목들 간의 상관 계수 구하기
# 수치형 데이터만 선택하여 상관계수 산출
correlations = train_clean.select_dtypes(include=[np.number]).corr()['smoking'].sort_values(ascending=False)

print("--- 'smoking' 항목과의 상관 계수 ---")
display(correlations)

## 추가 증빙 — BMI/나이/혈압 상관계수, 전체 히트맵

#### 추가 증빙: BMI/나이/수축기/이완기/HDL의 상관계수

In [ ]:
cols = ['BMI', 'age', 'systolic', 'relaxation', 'BMI', 'smoking']
correlation_matrix = train_clean[cols].corr()

print("--- BMI, 나이, 혈압 간의 상관계수 ---")
display(correlation_matrix)

In [ ]:
cols_to_corr = ['Gtp', 'ALT', 'AST', 'waist(cm)', 'age', 'BMI', 'smoking']
correlation_matrix_2 = train_clean[cols_to_corr].corr()

print("--- 간 수치, 허리둘레, 나이, BMI, 흡연 여부 간의 상관계수 ---")
display(correlation_matrix_2)

In [ ]:
# 모든 수치형 데이터 간의 상관계수 산출
full_correlation_matrix = train_clean.select_dtypes(include=[np.number]).corr()

print("--- 전체 지표별 상관계수 (상위 10개 행) ---")
display(full_correlation_matrix.head(10))

# 상관계수 히트맵 시각화
plt.figure(figsize=(15, 10))
sns.heatmap(full_correlation_matrix, annot=False, cmap='coolwarm', center=0)
plt.title('전체 지표 간 상관관계 히트맵')
plt.show()

### 나이, BMI, 흡연과 주요 지표 간의 상관관계 분석

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# train_clean이 정의되어 있는지 확인하고, 필요시 이전 단계의 필터링을 다시 적용합니다.
try:
    # 분석할 대상 컬럼 정의
    target_cols = ['age', 'BMI', 'smoking']
    # 수치형 데이터 전체에 대해 상관계수 계산
    corr_matrix = train_clean.select_dtypes(include=[np.number]).corr()

    # 나이, BMI, 흡연 여부에 대한 상관계수만 추출
    selected_corr = corr_matrix[target_cols].transpose()

    print("--- 나이, BMI, 흡연과 타 지표 간 상관계수 표 ---")
    display(selected_corr)

    # 히트맵 시각화
    plt.figure(figsize=(16, 6))
    sns.heatmap(selected_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
    plt.title('나이, BMI, 흡연과 다른 지표 간의 상관관계 히트맵')
    plt.show()
except NameError:
    print("오류: 'train_clean' 데이터프레임이 메모리에 없습니다. 데이터 로드 및 전처리 셀을 먼저 실행해 주세요.")